# Sensitivity of the neural models to training configuration

This notebook reproduces the sensitivity analysis of the neural models reported in the paper (Section 4.7, Table 26, Figure 21). It contains two experiments:

| Test | Question | Scope |
|---|---|---|
| **A. Sensitivity ladder** | Does the gap between the classical and the neural families change when the learning-rate schedule, regularization, or training duration of the neural models change? | 6 neural models, 17 datasets, Mode 4 (Borderline-SMOTE + SHAP-RF), one factor at a time, plus one combined small-data arm and an optimistic best-arm bound |
| **D. FT-Transformer × feature selection** | Is the decline of FT-Transformer under feature selection a property of its default configuration? | FT-Transformer, default vs. compact configuration, Modes 1 and 3, 17 datasets |

Every step reuses the logic of the main pipeline (`revised_comprehensive_software_defects.py`): data loading, de-duplication, 5-fold `StratifiedKFold(shuffle=True, random_state=42)`, training-fold median imputation, fold-wise SHAP-RF subsets (re-used from the pipeline output when available), Borderline-SMOTE (k=5, seed 42), the TabNet 85/15 split before SMOTE, and the PyTorch Tabular defaults. The outer folds and feature subsets are therefore the same as in the main results, and the classical reference is the published pipeline result for the same folds.

**Outputs** (all under `OUT_ROOT`): raw fold-level CSV files for each test, tables (`.csv` and `.tex`), and figures. Each runner skips work that is already saved, so a run can be stopped and resumed (e.g., after a Colab disconnect).

In [ ]:
!pip -q install pytorch_tabular pytorch-tabnet xgboost lightgbm catboost imbalanced-learn shap statsmodels

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab -- skipping Google Drive mount.")

## Data (same download and harmonization as the pipeline)

In [ ]:
# ==========================================================================
# DATA: identical download + harmonization as the main pipeline.
# Skipped automatically if DATA_DIR already contains the 17 .arff files.
# (Needs ~/.kaggle/kaggle.json, exactly as in the pipeline notebook.)
# ==========================================================================
import os, re, shutil
from glob import glob
import pandas as pd

DATA_DIR = os.environ.get("SDP_RAW_DATA_DIR", "/content/defect_data")

if os.path.isdir(DATA_DIR) and len(glob(os.path.join(DATA_DIR, "*.arff"))) > 0:
    print(f"{DATA_DIR} already prepared ({len(glob(os.path.join(DATA_DIR, '*.arff')))} .arff files; 17 expected).")
else:
    os.system("pip -q install kaggle")
    os.system("kaggle datasets download -d aczy156/software-defect-prediction-nasa")
    os.system("unzip -q -o software-defect-prediction-nasa.zip -d nasa_defect_data")
    os.system("kaggle datasets download -d faisalmohammad/aeeem-dataet4sdp")
    os.system("unzip -q -o aeeem-dataet4sdp.zip -d aeeem_data")

    # --- AEEEM csv -> arff (same code as the pipeline) ---
    os.makedirs("aeeem_data_arff", exist_ok=True)
    for csv_file in glob(os.path.join("aeeem_data", "*.csv")):
        df = pd.read_csv(csv_file)
        df.columns = [re.sub(r'[^A-Za-z0-9_]', '_', str(c)) for c in df.columns]
        df["Defective"] = (df["class"].astype(str).str.strip().str.lower()
                           .map({"buggy": "Y", "clean": "N"}))
        df = df.drop(columns=["class"])
        df = df[[c for c in df.columns if c != "Defective"] + ["Defective"]]
        name = os.path.splitext(os.path.basename(csv_file))[0]
        with open(os.path.join("aeeem_data_arff", name + ".arff"), "w", encoding="utf-8") as f:
            f.write(f"@relation {name}\n\n")
            for c in df.columns[:-1]:
                f.write(f"@attribute {c} numeric\n")
            f.write("@attribute Defective {Y,N}\n\n@data\n")
            for _, row in df.iterrows():
                f.write(",".join("?" if pd.isna(v) else str(v) for v in row) + "\n")

    # --- collect all .arff into DATA_DIR (same code as the pipeline) ---
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    os.makedirs(DATA_DIR, exist_ok=True)
    for src in ["nasa_defect_data", "aeeem_data_arff"]:
        for root, _, files in os.walk(src):
            for fn in files:
                if fn.lower().endswith(".arff"):
                    dst = os.path.join(DATA_DIR, fn)
                    k = 1
                    while os.path.exists(dst):
                        b, e = os.path.splitext(fn); dst = os.path.join(DATA_DIR, f"{b}_{k}{e}"); k += 1
                    shutil.copy2(os.path.join(root, fn), dst)
    print("Prepared:", sorted(os.listdir(DATA_DIR)))

## Configuration

In [ ]:
# ==========================================================================
# CONFIGURATION
# ==========================================================================
import os, gc, json, time, math, glob, zipfile, warnings, itertools, logging, shutil, tempfile
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
for _lg in ("pytorch_tabular", "lightning", "pytorch_lightning", "lightning.pytorch", "lightning_fabric"):
    logging.getLogger(_lg).setLevel(logging.ERROR)

# Root of the revised pipeline outputs (the four mode folders live here)
RESULTS_ROOT = os.environ.get("SDP_RESULTS_ROOT", "/content/drive/MyDrive/Software_Defects_Revised")
MODE_FOLDERS = {1: "results_no_smote", 2: "results_smote",
                3: "results_no_smote_feature_selection", 4: "results_smote_feature_selection"}

MODE = 4                       # mode used for Test A (Mode 4 = SMOTE + FS, the reference pipeline)
USE_SMOTE = MODE in (2, 4)
USE_FS = MODE in (3, 4)

OUT_ROOT = os.environ.get("SDP_R32_OUT", os.path.join(RESULTS_ROOT, "R3_2_sensitivity_tuning"))
os.makedirs(OUT_ROOT, exist_ok=True)

# ---- identical to the pipeline ----
RANDOM_STATE = 42
N_SPLITS = 5
DEDUPLICATE = True
MAX_EPOCHS = 100
DATASET_FILES = ["PC1.arff", "PC2.arff", "PC3.arff", "PC4.arff", "PC5.arff",
                 "JM1.arff", "KC1.arff", "KC3.arff",
                 "CM1.arff", "MC1.arff", "MC2.arff", "MW1.arff",
                 "EQ.arff", "JDT.arff", "LC.arff", "ML.arff", "PDE.arff"]
TARGET_CANDIDATES = ["Defective", "defects", "Defects", "DEFECT", "problems",
                     "class", "Class", "label", "Label"]
CLASSICAL_MODELS = ["LogisticRegression", "NaiveBayes", "RandomForest", "ExtraTrees",
                    "XGBoost", "LightGBM", "CatBoost"]
DEEP_TABULAR = ["TabNet", "GANDALF", "DANet"]
TRANSFORMER = ["FTTransformer", "TabTransformer", "AutoInt"]
NEURAL_MODELS = DEEP_TABULAR + TRANSFORMER
FAMILY = {**{m: "Classical" for m in CLASSICAL_MODELS}, **{m: "DeepTabular" for m in DEEP_TABULAR},
          **{m: "Transformer" for m in TRANSFORMER}}

# ---- which tests to run ----
RUN_TEST_A = True     # sensitivity ladder
RUN_TEST_D = True     # FT-Transformer x feature selection

# ---- TabNet batching ----
# TabNet is trained with drop_last=False, so the last (incomplete) mini-batch of
# every epoch is also used. With drop_last=True and batch_size=256, a training
# split with fewer than 256 rows would yield no complete batch.
TABNET_DROP_LAST = False

# ---- Test A: sensitivity ladder (one factor at a time) ----
# None = pipeline value. TabNet mappings are documented in NEURAL_KNOBS_DOC below.
LADDER_ARMS = {
    "default":         {},                                   # pipeline configuration
    "lr_plateau":      {"scheduler": True},                  # ReduceLROnPlateau
    "wd_1e-4":         {"weight_decay": 1e-4},
    "wd_1e-3":         {"weight_decay": 1e-3},
    "dropout_0.2":     {"dropout": 0.2},
    "dropout_0.4":     {"dropout": 0.4},
    "patience_15":     {"patience": 15},
    "batch_64":        {"batch_size": 64},
    "small_data":      {"batch_size": 64, "patience": 15, "scheduler": True},   # combined arm
}
LADDER_MODELS = NEURAL_MODELS
LADDER_DATASETS = None        # None = all 17

# ---- Test D ----
FT_COMPACT = {"num_attn_blocks": 2, "input_embed_dim": 32, "num_heads": 4}

FOLDS_TO_RUN = [1, 2, 3, 4, 5]

DATA_DIR = os.environ.get("SDP_RAW_DATA_DIR", "/content/defect_data")
AVAILABLE = [f for f in DATASET_FILES if os.path.exists(os.path.join(DATA_DIR, f))]
CKPT_DIR = tempfile.mkdtemp(prefix="pt_ckpt_")
print("RESULTS_ROOT:", RESULTS_ROOT)
print("OUT_ROOT    :", OUT_ROOT)
print("datasets available:", len(AVAILABLE), "| mode:", MODE)

NEURAL_KNOBS_DOC = """
Knob mapping (PyTorch Tabular models / TabNet):
  scheduler     : ReduceLROnPlateau on valid_loss (factor 0.5, patience 2) / ReduceLROnPlateau on val AUC (factor 0.5, patience 5)
  weight_decay  : Adam weight_decay / Adam weight_decay
  dropout       : model dropout fields (FT/TabTransformer: attn, ff, add_norm; AutoInt: attn_dropouts;
                  GANDALF: gflu_dropout; DANet: dropout_rate) / TabNet has no dropout -> lambda_sparse
                  (0.2 -> 1e-3, 0.4 -> 1e-2; pipeline 1e-4)
  patience      : early_stopping_patience 3 -> 15 / patience 20 -> 50
  batch_size    : 256 -> 64 / 256 -> 64 (virtual 128 -> 32)
"""
print(NEURAL_KNOBS_DOC)

## Pipeline-aligned helpers

In [ ]:
# ==========================================================================
# PIPELINE-ALIGNED HELPERS (copied from revised_comprehensive_software_defects.py)
# ==========================================================================
from scipy.io import arff
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef)
from imblearn.over_sampling import BorderlineSMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import shap
import torch

try:   # PyTorch >= 2.6 weights_only fix (same as pipeline)
    import omegaconf, omegaconf.dictconfig, omegaconf.listconfig, omegaconf.base, omegaconf.nodes
    torch.serialization.add_safe_globals([
        omegaconf.dictconfig.DictConfig, omegaconf.listconfig.ListConfig,
        omegaconf.base.ContainerMetadata, omegaconf.base.Metadata, omegaconf.nodes.AnyNode,
        omegaconf.nodes.IntegerNode, omegaconf.nodes.FloatNode, omegaconf.nodes.StringNode,
        omegaconf.nodes.BooleanNode, omegaconf.nodes.EnumNode, omegaconf.nodes.InterpolationResultNode])
except Exception:
    pass

from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabular import TabularModel
from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig
from pytorch_tabular.models.ft_transformer.config import FTTransformerConfig
from pytorch_tabular.models.tab_transformer.config import TabTransformerConfig
from pytorch_tabular.models.autoint.config import AutoIntConfig
from pytorch_tabular.models.gandalf.config import GANDALFConfig
from pytorch_tabular.models.danet.config import DANetConfig

GPU_AVAILABLE = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU_AVAILABLE else "none (CPU)")


def _map_target_value(v):
    if isinstance(v, bytes):
        v = v.decode("utf-8")
    if isinstance(v, str):
        v_low = v.strip().lower()
        if v_low in ("y", "yes", "true", "1", "defective", "true_y"):
            return 1
        if v_low in ("n", "no", "false", "0", "not_defective", "false_y"):
            return 0
        try:
            return float(v_low)
        except ValueError:
            return np.nan
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan


def load_dataset(path):
    """Identical to the pipeline's load_dataset (incl. exact-duplicate removal)."""
    if path.lower().endswith(".arff"):
        data, _ = arff.loadarff(path)
        df = pd.DataFrame(data)
    else:
        df = pd.read_csv(path)
    for col in df.columns:
        if df[col].dtype == object:
            try:
                df[col] = df[col].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)
            except Exception:
                pass
    target_col = next((c for c in TARGET_CANDIDATES if c in df.columns), df.columns[-1])
    df[target_col] = df[target_col].apply(_map_target_value)
    df = df.rename(columns={target_col: "target"}).dropna(subset=["target"])
    df["target"] = df["target"].astype(int)
    for col in df.columns:
        if col != "target":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(axis=1, how="all")
    if DEDUPLICATE:
        df = df.drop_duplicates().reset_index(drop=True)
    return df


def compute_metrics(y_true, y_pred, y_prob=None):
    out = {"Accuracy": accuracy_score(y_true, y_pred),
           "Precision": precision_score(y_true, y_pred, zero_division=0),
           "Recall": recall_score(y_true, y_pred, zero_division=0),
           "F1": f1_score(y_true, y_pred, zero_division=0),
           "MCC": matthews_corrcoef(y_true, y_pred)}
    try:
        out["ROC_AUC"] = roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan
    except Exception:
        out["ROC_AUC"] = np.nan
    return out


class _TimedBlock:
    def __enter__(self):
        if GPU_AVAILABLE:
            torch.cuda.synchronize()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *a):
        if GPU_AVAILABLE:
            torch.cuda.synchronize()
        self.seconds = time.perf_counter() - self._t0
        return False


def select_features_fold_core(X_train, y_train):
    """Computation of the pipeline's select_features_fold() without the plots."""
    X_train = X_train.copy().apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
    X_train = X_train.fillna(X_train.median()).fillna(0)
    y_train = pd.Series(y_train).copy()
    Xs = pd.DataFrame(StandardScaler().fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    rf = RandomForestClassifier(n_estimators=300, max_depth=None, class_weight="balanced",
                                random_state=42, n_jobs=-1).fit(Xs, y_train)
    rf_imp = pd.Series(rf.feature_importances_, index=X_train.columns)
    rf_sel = rf_imp[rf_imp >= np.percentile(rf_imp.values, 50)].index.tolist()
    bg = Xs.sample(n=100, random_state=42) if len(Xs) > 100 else Xs
    sv = shap.TreeExplainer(rf, data=bg).shap_values(Xs, check_additivity=False)
    if isinstance(sv, list):
        arr = sv[1] if len(sv) == 2 else sv[0]
    else:
        arr = np.asarray(sv)
        if arr.ndim == 3:
            arr = arr[:, :, 1] if arr.shape[2] == 2 else arr[:, :, 0]
    sh_imp = pd.Series(np.mean(np.abs(arr), axis=0), index=X_train.columns)
    sh_sel = sh_imp[sh_imp >= np.percentile(sh_imp.values, 50)].index.tolist()
    cons = [f for f in X_train.columns if f in rf_sel and f in sh_sel]
    if not cons:
        cons = [(rf_imp.rank(pct=True) + sh_imp.rank(pct=True)).idxmax()]
    return cons


CLASSICAL_CTORS = {   # identical to the pipeline
    "LogisticRegression": lambda: LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    "NaiveBayes": lambda: GaussianNB(),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=500, max_depth=20, class_weight="balanced",
                                                   random_state=RANDOM_STATE),
    "ExtraTrees": lambda: ExtraTreesClassifier(n_estimators=500, class_weight="balanced", random_state=RANDOM_STATE),
    "XGBoost": lambda: XGBClassifier(n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8,
                                     colsample_bytree=0.8, eval_metric="logloss", random_state=RANDOM_STATE),
    "LightGBM": lambda: LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=RANDOM_STATE, verbose=-1),
    "CatBoost": lambda: CatBoostClassifier(iterations=500, learning_rate=0.05, depth=8, verbose=0),
}
PT_CONFIG_CLASS = {"FTTransformer": FTTransformerConfig, "TabTransformer": TabTransformerConfig,
                   "AutoInt": AutoIntConfig, "GANDALF": GANDALFConfig, "DANet": DANetConfig}
PT_DROPOUT_FIELDS = {"FTTransformer": ["attn_dropout", "ff_dropout", "add_norm_dropout"],
                     "TabTransformer": ["attn_dropout", "ff_dropout", "add_norm_dropout"],
                     "AutoInt": ["attn_dropouts"], "GANDALF": ["gflu_dropout"], "DANet": ["dropout_rate"]}
TABNET_DEFAULT = dict(n_d=32, n_a=32, n_steps=5, gamma=1.5, lambda_sparse=1e-4, lr=2e-2, mask_type="entmax")
TABNET_REG_MAP = {0.2: 1e-3, 0.4: 1e-2}

## Outer folds and fold-wise feature subsets (identical to the pipeline)

In [ ]:
# ==========================================================================
# OUTER FOLDS identical to the pipeline
#   load -> dedup -> StratifiedKFold(5, shuffle, 42) -> training-median imputation
#   -> fold-wise SHAP-RF subset (read from the pipeline output when available,
#      so the features are exactly those of the published results)
# ==========================================================================
FS_CACHE_DIR = os.path.join(OUT_ROOT, "fs_subsets")
os.makedirs(FS_CACHE_DIR, exist_ok=True)


def _pipeline_subset(ds, fold, mode):
    """Read selected_features.txt written by the pipeline (folder or zip)."""
    rel = f"feature_selection/fold_{fold}/selected_features.txt"
    for m in ([mode] if mode in (3, 4) else []) + [4, 3]:
        base = os.path.join(RESULTS_ROOT, MODE_FOLDERS[m], ds)
        p = os.path.join(base, rel)
        if os.path.exists(p):
            return [l.strip() for l in open(p) if l.strip()]
        zp = base + ".zip"
        if os.path.exists(zp):
            with zipfile.ZipFile(zp) as z:
                hits = [n for n in z.namelist() if n.replace("\\", "/").endswith(rel)]
                if hits:
                    return [l.strip() for l in z.read(hits[0]).decode().splitlines() if l.strip()]
    return None


def get_fs_subset(ds, fold, X_tr, y_tr, mode):
    cache = os.path.join(FS_CACHE_DIR, f"{ds}_fold{fold}.json")
    if os.path.exists(cache):
        return json.load(open(cache))["features"]
    feats, src = _pipeline_subset(ds, fold, mode), "pipeline"
    if feats is None or not set(feats) <= set(X_tr.columns):
        feats, src = select_features_fold_core(X_tr, y_tr), "recomputed"
    json.dump({"features": feats, "source": src}, open(cache, "w"))
    return feats


_FOLD_CACHE = {}


def get_folds(ds, mode=MODE):
    """List of dicts (fold, Xtr, ytr, Xte, yte) for one dataset, exactly as in the pipeline."""
    key = (ds, mode)
    if key in _FOLD_CACHE:
        return _FOLD_CACHE[key]
    df = load_dataset(os.path.join(DATA_DIR, ds + ".arff"))
    X = df.drop(columns=["target"]).reset_index(drop=True)
    y = df["target"].reset_index(drop=True)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    out = []
    for k, (tr, te) in enumerate(skf.split(X, y), start=1):
        Xtr, Xte = X.iloc[tr].reset_index(drop=True), X.iloc[te].reset_index(drop=True)
        ytr, yte = y.iloc[tr].reset_index(drop=True), y.iloc[te].reset_index(drop=True)
        med = Xtr.median()
        Xtr, Xte = Xtr.fillna(med).fillna(0), Xte.fillna(med).fillna(0)
        if mode in (3, 4):
            S = get_fs_subset(ds, k, Xtr, ytr, mode)
            Xtr, Xte = Xtr[S].copy(), Xte[S].copy()
        out.append(dict(fold=k, Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte))
    _FOLD_CACHE[key] = out
    return out


DATASETS = [os.path.splitext(f)[0] for f in AVAILABLE]
if USE_FS:
    src = []
    for ds in DATASETS:
        for f in get_folds(ds):
            pass
        src += [json.load(open(os.path.join(FS_CACHE_DIR, f"{ds}_fold{k}.json")))["source"] for k in range(1, N_SPLITS + 1)]
    print(f"Fold-wise feature subsets: {src.count('pipeline')} read from the pipeline output, "
          f"{src.count('recomputed')} recomputed (expected: all from the pipeline).")

## Configurable model runners (`cfg={}` reproduces the pipeline)

In [ ]:
# ==========================================================================
# MODEL RUNNERS
#   cfg = {} reproduces the pipeline exactly. Keys (all optional):
#     lr, batch_size, patience, weight_decay, scheduler (bool), dropout (float),
#     arch (dict of model-config overrides), tabnet (dict: n_d, n_a, n_steps,
#     gamma, lambda_sparse), max_epochs
# ==========================================================================
import torch.optim.lr_scheduler as _lrs


def maybe_smote(X, y, use_smote):
    if not use_smote:
        return X, y
    try:
        return BorderlineSMOTE(random_state=RANDOM_STATE, k_neighbors=5).fit_resample(X, y)
    except Exception:
        return X, y          # same fallback as the pipeline


def _cleanup():
    gc.collect()
    if GPU_AVAILABLE:
        torch.cuda.empty_cache()


def fit_predict_pt(name, cfg, Xtr, ytr, Xte, use_smote):
    """PyTorch Tabular model, same protocol as the pipeline: SMOTE on the training
    partition, then TabularModel.fit(train=...) with its internal 20% validation split."""
    Xf, yf = maybe_smote(Xtr, ytr, use_smote)
    fit_df = Xf.copy(); fit_df["target"] = np.asarray(yf)
    mk = dict(task="classification", learning_rate=cfg.get("lr") or 1e-3)
    if cfg.get("dropout") is not None:
        for fld in PT_DROPOUT_FIELDS[name]:
            mk[fld] = cfg["dropout"]
    mk.update(cfg.get("arch", {}))
    ok = {}
    if cfg.get("weight_decay"):
        ok["optimizer_params"] = {"weight_decay": cfg["weight_decay"]}
    if cfg.get("scheduler"):
        ok.update(lr_scheduler="ReduceLROnPlateau",
                  lr_scheduler_params={"mode": "min", "factor": 0.5, "patience": 2},
                  lr_scheduler_monitor_metric="valid_loss")
    tk = dict(batch_size=cfg.get("batch_size") or 256, max_epochs=cfg.get("max_epochs") or MAX_EPOCHS,
              progress_bar="none", checkpoints_path=CKPT_DIR)
    if cfg.get("patience"):
        tk["early_stopping_patience"] = cfg["patience"]
    if not GPU_AVAILABLE:
        tk["accelerator"] = "cpu"
    tm = TabularModel(data_config=DataConfig(target=["target"], continuous_cols=list(Xf.columns)),
                      model_config=PT_CONFIG_CLASS[name](**mk), optimizer_config=OptimizerConfig(**ok),
                      trainer_config=TrainerConfig(**tk), verbose=False, suppress_lightning_logger=True)
    with _TimedBlock() as t_tr:
        tm.fit(train=fit_df)
    te = Xte.copy(); te["target"] = 0          # placeholder, not used for prediction
    with _TimedBlock() as t_in:
        pdf = tm.predict(te)
    prob = pdf[sorted(c for c in pdf.columns if c.endswith("_probability"))[-1]].values
    pred = pdf[[c for c in pdf.columns if c.endswith("prediction")][0]].values.astype(int)
    try:
        n_epochs = int(tm.model.current_epoch) + 1 if hasattr(tm, "model") else np.nan
    except Exception:
        n_epochs = np.nan
    del tm; _cleanup()
    return prob, pred, t_tr.seconds, t_in.seconds, n_epochs


def fit_predict_tabnet(cfg, Xtr, ytr, Xte, use_smote):
    """TabNet, same protocol as the pipeline: 15% stratified validation split BEFORE
    SMOTE, SMOTE on the rest, early stopping on the validation set."""
    Xa, Xv, ya, yv = train_test_split(Xtr, ytr, test_size=0.15, stratify=ytr, random_state=RANDOM_STATE)
    Xa, ya = maybe_smote(Xa, ya, use_smote)
    p = dict(TABNET_DEFAULT); p.update(cfg.get("tabnet", {}))
    if cfg.get("lr"):
        p["lr"] = cfg["lr"]
    if cfg.get("dropout") is not None:
        p["lambda_sparse"] = TABNET_REG_MAP.get(cfg["dropout"], cfg["dropout"])
    opt = dict(lr=p.pop("lr"))
    if cfg.get("weight_decay"):
        opt["weight_decay"] = cfg["weight_decay"]
    kw = dict(optimizer_params=opt, verbose=0, **p)
    if cfg.get("scheduler"):
        kw.update(scheduler_fn=_lrs.ReduceLROnPlateau, scheduler_params={"mode": "max", "factor": 0.5, "patience": 5})
    bs = cfg.get("batch_size") or 256
    patience = {None: 20, 15: 50}.get(cfg.get("patience"), cfg.get("patience"))
    m = TabNetClassifier(**kw)
    with _TimedBlock() as t_tr:
        m.fit(Xa.values.astype(np.float32), np.asarray(ya),
              eval_set=[(Xv.values.astype(np.float32), np.asarray(yv))],
              max_epochs=cfg.get("max_epochs") or MAX_EPOCHS, patience=patience,
              batch_size=bs, virtual_batch_size=max(bs // 2, 16),
              drop_last=cfg.get("drop_last", TABNET_DROP_LAST))
    Xt = Xte.values.astype(np.float32)
    with _TimedBlock() as t_in:
        prob = m.predict_proba(Xt)[:, 1]
        pred = m.predict(Xt).astype(int)
    n_epochs = len(m.history["loss"])
    zero_steps = bool(np.allclose(m.history["loss"], 0.0))
    del m; _cleanup()
    return prob, pred, t_tr.seconds, t_in.seconds, n_epochs, zero_steps, len(Xa)


def run_neural(name, cfg, Xtr, ytr, Xte, yte, use_smote):
    """Fit one neural model and return a metrics dict."""
    if name == "TabNet":
        prob, pred, ttr, tin, ep, zero, n_fit = fit_predict_tabnet(cfg, Xtr, ytr, Xte, use_smote)
    else:
        prob, pred, ttr, tin, ep = fit_predict_pt(name, cfg, Xtr, ytr, Xte, use_smote)
        zero, n_fit = False, np.nan
    m = compute_metrics(np.asarray(yte), pred, prob)
    m.update(train_time_sec=ttr, infer_time_sec=tin, epochs=ep, zero_grad_steps=zero, n_fit_rows=n_fit)
    return m


def run_classical(name, Xtr, ytr, Xte, yte, use_smote):
    Xf, yf = maybe_smote(Xtr, ytr, use_smote)
    model = CLASSICAL_CTORS[name]()
    with _TimedBlock() as t_tr:
        model.fit(Xf, yf)
    with _TimedBlock() as t_in:
        prob = model.predict_proba(Xte)[:, 1]
        pred = model.predict(Xte)
    m = compute_metrics(np.asarray(yte), pred, prob)
    m.update(train_time_sec=t_tr.seconds, infer_time_sec=t_in.seconds)
    return m


# ---------- resumable result store ----------
def result_store(name):
    path = os.path.join(OUT_ROOT, name)
    done = set()
    if os.path.exists(path):
        done = set(pd.read_csv(path)["key"])
    return path, done


RESULT_SCHEMA = ["key", "Mode", "Arm", "Variant", "Config", "Dataset", "Model", "Family", "Fold", "status",
                 "Accuracy", "Precision", "Recall", "F1", "MCC", "ROC_AUC", "train_time_sec", "infer_time_sec",
                 "epochs", "zero_grad_steps", "n_fit_rows", "inner_auc", "n_trials", "n_pruned",
                 "tune_time_sec", "best_params"]


def append_row(path, row):
    row = dict(row)
    row.setdefault("Family", FAMILY.get(row.get("Model"), ""))
    row.setdefault("status", "ok")
    pd.DataFrame([row]).reindex(columns=RESULT_SCHEMA).to_csv(
        path, mode="a", header=not os.path.exists(path), index=False)

## Classical reference (published defaults on the same folds)

In [ ]:
# ==========================================================================
# REFERENCE: published fold-level results of the pipeline (same folds)
#   The classical models are used with their default (published) results.
#   If the pipeline CSV is not available, the classical models are recomputed
#   on exactly the same folds.
# ==========================================================================
def load_published(mode):
    p = os.path.join(RESULTS_ROOT, MODE_FOLDERS[mode], "ALL_DATASETS_foldwise_metrics.csv")
    if os.path.exists(p):
        d = pd.read_csv(p)
        return d.drop_duplicates(subset=["Dataset", "Model", "Fold"], keep="last")
    return None


PUBLISHED = {m: load_published(m) for m in MODE_FOLDERS}
for m, d in PUBLISHED.items():
    print(f"Mode {m}: published fold-level results", "found" if d is not None else "NOT found")

CLASSICAL_REF_PATH = os.path.join(OUT_ROOT, f"classical_reference_mode{MODE}.csv")


def classical_reference(mode=MODE):
    pub = PUBLISHED.get(mode)
    if pub is not None and set(CLASSICAL_MODELS) <= set(pub["Model"]):
        ref = pub[pub["Model"].isin(CLASSICAL_MODELS)][["Dataset", "Model", "Fold", "ROC_AUC", "F1", "MCC",
                                                        "train_time_sec"]].copy()
        ref["source"] = "published"
        return ref
    path, done = result_store(os.path.basename(CLASSICAL_REF_PATH))
    for ds in DATASETS:
        for f in get_folds(ds, mode):
            if f["fold"] not in FOLDS_TO_RUN:
                continue
            for mdl in CLASSICAL_MODELS:
                key = f"{ds}|{mdl}|{f['fold']}"
                if key in done:
                    continue
                r = run_classical(mdl, f["Xtr"], f["ytr"], f["Xte"], f["yte"], mode in (2, 4))
                append_row(path, dict(key=key, Dataset=ds, Model=mdl, Fold=f["fold"], **r))
    ref = pd.read_csv(path)
    ref["source"] = "recomputed"
    return ref


CLASSICAL_REF = classical_reference()
print(f"Classical reference: {len(CLASSICAL_REF)} fold-level rows ({CLASSICAL_REF['source'].iloc[0]}).")

## Test A — Sensitivity ladder (scheduler, regularization, training duration)

In [ ]:
# ==========================================================================
# TEST A -- SENSITIVITY LADDER (one factor at a time, + one combined arm)
#   6 neural models x 17 datasets x 5 outer folds x arms, Mode 4.
#   Classical models stay at their published defaults (CLASSICAL_REF).
# ==========================================================================
# ---- approximate budget from the published Mode-4 training times ----
ARM_COST_FACTOR = {"default": 1, "lr_plateau": 1.3, "wd_1e-4": 1, "wd_1e-3": 1, "dropout_0.2": 1,
                   "dropout_0.4": 1, "patience_15": 3, "batch_64": 2.5, "small_data": 5}
pub = PUBLISHED.get(MODE)
if pub is not None:
    t = pub[pub["Model"].isin(LADDER_MODELS)].groupby("Model")["train_time_sec"].sum()  # all datasets x folds
    est = sum(t.sum() * ARM_COST_FACTOR.get(a, 2) for a in LADDER_ARMS) / 3600
    print(f"Test A approximate budget: {est:.1f} GPU-hours for {len(LADDER_ARMS)} arms "
          f"(approximate per-arm cost factors).")

TA_PATH, TA_DONE = result_store(f"testA_ladder_mode{MODE}.csv")
ladder_ds = LADDER_DATASETS or DATASETS


# ---- progress summary: done / not done ----
def testA_progress(tag):
    expected = {f"{arm}|{ds}|{mdl}|{fold}" for ds in ladder_ds for mdl in LADDER_MODELS
                for fold in FOLDS_TO_RUN for arm in LADDER_ARMS}
    n_all = len(expected)
    n_done = len(expected & TA_DONE)
    print(f"[Test A {tag}] done {n_done}/{n_all} ({100 * n_done / n_all:.1f}%) | "
          f"not done {n_all - n_done} ({100 * (n_all - n_done) / n_all:.1f}%)")

testA_progress("before run")

In [ ]:
if RUN_TEST_A:
    for ds in ladder_ds:
        folds = get_folds(ds, MODE)
        t0 = time.time()
        for mdl in LADDER_MODELS:
            for f in folds:
                if f["fold"] not in FOLDS_TO_RUN:
                    continue
                for arm, cfg in LADDER_ARMS.items():
                    key = f"{arm}|{ds}|{mdl}|{f['fold']}"
                    if key in TA_DONE:
                        continue
                    try:
                        r = run_neural(mdl, cfg, f["Xtr"], f["ytr"], f["Xte"], f["yte"], USE_SMOTE)
                        status = "ok"
                    except Exception as e:
                        r, status = {}, f"error: {type(e).__name__}: {str(e)[:120]}"
                    append_row(TA_PATH, dict(key=key, Arm=arm, Dataset=ds, Model=mdl, Fold=f["fold"],
                                             status=status, **r))
                    TA_DONE.add(key)
        print(f"  Test A: {ds} done in {(time.time() - t0) / 60:.1f} min")
    print("Test A complete:", TA_PATH)

## Test D — FT-Transformer × feature selection

In [ ]:
# ==========================================================================
# TEST D -- FT-TRANSFORMER x FEATURE SELECTION
#   FT-Transformer is run in Mode 1 (all features) and Mode 3 (fold-wise SHAP-RF
#   subset), both without SMOTE, with (a) the pipeline default (6 blocks, 8 heads,
#   32-d tokens, batch 256, patience 3) and (b) a compact, small-data configuration
#   (2 blocks, 4 heads, batch 64, patience 15). If the FS decline is a property of
#   the default configuration, it should shrink or vanish with (b).
# ==========================================================================
TD_CONFIGS = {"default": {},
              "compact_small_data": {"arch": FT_COMPACT, "batch_size": 64, "patience": 15}}
TD_PATH, TD_DONE = result_store("testD_ft_transformer_fs.csv")


# ---- progress summary: done / not done ----
def testD_progress(tag):
    expected = {
        f"{mode}|{ds}|{cname}|{fold}"
        for ds in DATASETS
        for mode in (1, 3)
        for cname in TD_CONFIGS
        for fold in FOLDS_TO_RUN
    }

    n_all = len(expected)
    n_done = len(expected & TD_DONE)
    pct_done = 100 * n_done / n_all if n_all else 0.0
    pct_left = 100 - pct_done if n_all else 0.0

    print(
        f"[Test D {tag}] done {n_done}/{n_all} ({pct_done:.1f}%) | "
        f"not done {n_all - n_done} ({pct_left:.1f}%)"
    )


testD_progress("before run")

In [ ]:
if RUN_TEST_D:
    for ds in DATASETS:
        for mode in (1, 3):
            for f in get_folds(ds, mode):
                if f["fold"] not in FOLDS_TO_RUN:
                    continue
                for cname, cfg in TD_CONFIGS.items():
                    key = f"{mode}|{ds}|{cname}|{f['fold']}"
                    if key in TD_DONE:
                        continue
                    r = run_neural("FTTransformer", cfg, f["Xtr"], f["ytr"], f["Xte"], f["yte"], False)
                    append_row(TD_PATH, dict(key=key, Mode=mode, Dataset=ds, Model="FTTransformer",
                                             Config=cname, Fold=f["fold"], **r))
                    TD_DONE.add(key)
        print(f"  Test D: {ds} done")
    print("Test D complete:", TD_PATH)

# Analysis
## Statistical helpers

In [ ]:
# ==========================================================================
# ANALYSIS HELPERS (same procedures as the main statistical analysis)
# ==========================================================================
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib
import matplotlib.pyplot as plt

TAB_DIR = os.path.join(OUT_ROOT, "tables"); FIG_DIR = os.path.join(OUT_ROOT, "figures")
os.makedirs(TAB_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
N_BOOT = 5000
DISPLAY = {"FTTransformer": "FT-Transformer", "TabTransformer": "TabTransformer", "AutoInt": "AutoInt",
           "TabNet": "TabNet", "GANDALF": "GANDALF", "DANet": "DANet"}


def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = ~np.isnan(p)
    if ok.sum():
        out[ok] = multipletests(p[ok], method="holm")[1]
    return out


def cliffs(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    return ((x[:, None] > y[None, :]).sum() - (x[:, None] < y[None, :]).sum()) / (len(x) * len(y))


def boot_ci(v, seed=RANDOM_STATE):
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    if len(v) < 2:
        return np.nan, np.nan
    b = v[np.random.default_rng(seed).integers(0, len(v), (N_BOOT, len(v)))].mean(1)
    return np.percentile(b, 2.5), np.percentile(b, 97.5)


def cluster_boot_ci(df, value, cluster="Dataset", seed=RANDOM_STATE):
    """CI for a mean of fold-level values, resampling datasets (clusters) then folds."""
    rng = np.random.default_rng(seed)
    groups = [g[value].dropna().values for _, g in df.groupby(cluster)]
    groups = [g for g in groups if len(g)]
    if len(groups) < 2:
        return np.nan, np.nan
    means = []
    for _ in range(N_BOOT):
        pick = rng.integers(0, len(groups), len(groups))
        means.append(np.mean(np.concatenate([groups[i][rng.integers(0, len(groups[i]), len(groups[i]))] for i in pick])))
    return np.percentile(means, 2.5), np.percentile(means, 97.5)


def paired(diff):
    """Summary of paired differences (one value per dataset or per fold)."""
    d = np.asarray(diff, float); d = d[~np.isnan(d)]
    lo, hi = boot_ci(d)
    try:
        p = stats.wilcoxon(d).pvalue if np.any(np.abs(d) > 1e-12) and len(d) >= 2 else 1.0
    except ValueError:
        p = np.nan
    return dict(n=len(d), mean=d.mean() if len(d) else np.nan, ci_low=lo, ci_high=hi,
                pos=int((d > 1e-9).sum()), neg=int((d < -1e-9).sum()), p=p)


def pp(x, sign=True):
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else (f"{100 * x:+.2f}" if sign else f"{100 * x:.2f}")


def fmt_p(p):
    return "n/a" if p is None or np.isnan(p) else ("<0.001" if p < 0.001 else f"{p:.3f}")


def save_table(df, name, caption, label, index=False):
    df.to_csv(os.path.join(TAB_DIR, f"{name}.csv"), index=index)
    try:
        tex = df.to_latex(index=index, float_format="%.2f", caption=caption, label=label, escape=True)
        open(os.path.join(TAB_DIR, f"{name}.tex"), "w").write(tex)
    except Exception as e:
        print("LaTeX export skipped:", e)
    return df


def classical_fold_mean(mode=MODE):
    """Mean ROC-AUC of the 7 classical learners per (dataset, fold) -- the reference."""
    ref = CLASSICAL_REF if mode == MODE else PUBLISHED[mode][PUBLISHED[mode]["Model"].isin(CLASSICAL_MODELS)]
    return ref.groupby(["Dataset", "Fold"])["ROC_AUC"].mean().rename("Classical")


DS_SIZE = {ds: len(load_dataset(os.path.join(DATA_DIR, ds + ".arff"))) for ds in DATASETS}
SMALL = [d for d, n in DS_SIZE.items() if n < 400]
print("small datasets (< 400 instances):", SMALL)

## Results of Test A

In [ ]:
# ==========================================================================
# ANALYSIS A -- sensitivity ladder
#   gap(arm, family, dataset) = mean classical ROC-AUC (published defaults)
#                               - mean ROC-AUC of the family's models under the arm
#   shift = gap(arm) - gap(default arm, re-run)   (negative = arm narrows the gap)
#   Tests: Wilcoxon over the 17 datasets, Holm over all arm x family tests.
# ==========================================================================
TA = pd.read_csv(TA_PATH) if os.path.exists(TA_PATH) else pd.DataFrame()
A_GAP = A_MODEL = None
if len(TA):
    TA = TA[TA["status"] == "ok"]
    C_ds = classical_fold_mean(MODE).groupby("Dataset").mean()
    m_ds = TA.groupby(["Arm", "Model", "Dataset"])["ROC_AUC"].mean().reset_index()
    m_ds["Family"] = m_ds["Model"].map(FAMILY)
    fam = m_ds.groupby(["Arm", "Family", "Dataset"])["ROC_AUC"].mean().unstack("Dataset")

    # optimistic bound: best arm per (model, dataset) chosen on the TEST folds
    best = m_ds.groupby(["Model", "Dataset"])["ROC_AUC"].max().reset_index()
    best["Family"] = best["Model"].map(FAMILY)
    oracle = best.groupby(["Family", "Dataset"])["ROC_AUC"].mean().unstack("Dataset")

    arms = [a for a in LADDER_ARMS if a in fam.index.get_level_values(0)]
    rows = []
    for famname in ["DeepTabular", "Transformer"]:
        if ("default", famname) not in fam.index:
            continue
        g_def = C_ds - fam.loc[("default", famname)]
        for arm in arms + ["best_arm_per_dataset (oracle)"]:
            row_f = oracle.loc[famname] if arm.startswith("best") else fam.loc[(arm, famname)]
            g = (C_ds - row_f).dropna()
            s_gap = paired(g.values)
            sh = (g - g_def.reindex(g.index)).dropna()
            s_sh = paired(sh.values) if arm != "default" else dict(mean=0.0, ci_low=np.nan, ci_high=np.nan, p=np.nan, pos=0, neg=0)
            rows.append(dict(Family=famname, Arm=arm, n_datasets=s_gap["n"], gap=100 * s_gap["mean"],
                             gap_ci_low=100 * s_gap["ci_low"], gap_ci_high=100 * s_gap["ci_high"],
                             classical_better=s_gap["pos"], gap_p=s_gap["p"],
                             shift=100 * s_sh["mean"], shift_ci_low=100 * s_sh["ci_low"], shift_ci_high=100 * s_sh["ci_high"],
                             narrowed_on=s_sh["neg"], shift_p=s_sh["p"],
                             gap_small_ds=100 * g.reindex([d for d in SMALL if d in g.index]).mean(),
                             gap_other_ds=100 * g.reindex([d for d in g.index if d not in SMALL]).mean()))
    A_GAP = pd.DataFrame(rows)
    A_GAP["gap_p_holm"] = holm(A_GAP["gap_p"].values)
    A_GAP["shift_p_holm"] = holm(A_GAP["shift_p"].values)
    save_table(A_GAP.round(4), "Table_R32_ladder_gaps",
               "Sensitivity ladder (Mode 4): Classical $-$ family gap in ROC-AUC (percentage points; positive = classical better) for each training-configuration arm, its shift relative to the default arm, and Holm-adjusted Wilcoxon tests over datasets. The last arm selects, for every model and dataset, the best arm on the test folds and is therefore an optimistic bound.",
               "tab:r32_ladder")
    print(A_GAP[["Family", "Arm", "gap", "gap_ci_low", "gap_ci_high", "classical_better", "gap_p_holm",
                 "shift", "shift_ci_low", "shift_ci_high", "shift_p_holm"]].round(2).to_string(index=False))

    # per model x arm
    A_MODEL = (m_ds.groupby(["Model", "Arm"])["ROC_AUC"].mean().unstack("Arm")[arms] * 100)
    A_MODEL = A_MODEL.reindex([m for m in LADDER_MODELS if m in A_MODEL.index])
    for a in arms[1:]:
        A_MODEL[f"d_{a}"] = A_MODEL[a] - A_MODEL["default"]
    A_MODEL.index = A_MODEL.index.map(lambda m: DISPLAY.get(m, m))
    save_table(A_MODEL.round(2).reset_index(), "Table_R32_ladder_models",
               "Mean ROC-AUC (\\%) of each neural model under each ladder arm (Mode 4, 17 datasets x 5 folds) and the change relative to the default arm.",
               "tab:r32_ladder_models")
    ep = TA.groupby(["Arm", "Model"])["epochs"].mean().unstack("Arm")
    ep.round(1).to_csv(os.path.join(TAB_DIR, "Table_R32_ladder_epochs.csv"))

    # figure: gap per arm with CI
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    for ax, famname, col in zip(axes, ["DeepTabular", "Transformer"], ["#DD8452", "#55A868"]):
        g = A_GAP[A_GAP["Family"] == famname].reset_index(drop=True)
        y = np.arange(len(g))[::-1]
        ax.errorbar(g["gap"], y, xerr=[g["gap"] - g["gap_ci_low"], g["gap_ci_high"] - g["gap"]], fmt="o", color=col, capsize=3)
        ax.axvline(0, color="k", ls="--", lw=0.8)
        d0 = g.loc[g["Arm"] == "default", "gap"]
        if len(d0):
            ax.axvline(d0.iloc[0], color=col, ls=":", lw=0.8)
        ax.set_yticks(y); ax.set_yticklabels(g["Arm"]); ax.set_title(f"Classical $-$ {famname} (pp)")
        ax.set_xlabel("ROC-AUC gap, mean over datasets with 95% CI")
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "Fig_R32_ladder_gaps.png"), dpi=300); plt.close(fig)

## Results of Test D

In [ ]:
# ==========================================================================
# ANALYSIS D -- FT-Transformer x feature selection (Modes 1 vs 3, no SMOTE)
# ==========================================================================
TD = pd.read_csv(TD_PATH) if os.path.exists(TD_PATH) else pd.DataFrame()
D_TABLE = None
if len(TD):
    p = TD.pivot_table(index=["Config", "Dataset"], columns="Mode", values="ROC_AUC", aggfunc="mean")
    rows, eff = [], {}
    for cfg in p.index.get_level_values(0).unique():
        d = p.loc[cfg]
        fs = (d[3] - d[1]).dropna(); eff[cfg] = fs
        s = paired(fs.values)
        rows.append(dict(Config=cfg, n_datasets=s["n"], ROC_all_features=100 * d[1].mean(), ROC_FS=100 * d[3].mean(),
                         FS_effect=100 * s["mean"], ci_low=100 * s["ci_low"], ci_high=100 * s["ci_high"],
                         datasets_improved=s["pos"], datasets_declined=s["neg"], p=s["p"]))
    D_TABLE = pd.DataFrame(rows)
    if "default" in eff and "compact_small_data" in eff:
        dd = (eff["compact_small_data"] - eff["default"]).dropna()
        s = paired(dd.values)
        D_TABLE.loc[len(D_TABLE)] = dict(Config="difference (compact - default)", n_datasets=s["n"], FS_effect=100 * s["mean"],
                                         ci_low=100 * s["ci_low"], ci_high=100 * s["ci_high"], p=s["p"])
    D_TABLE["p_holm"] = holm(D_TABLE["p"].values)
    save_table(D_TABLE.round(3), "Table_R32_ft_fs",
               "FT-Transformer without oversampling: ROC-AUC (\\%) with all features (Mode 1) and with the fold-wise SHAP-RF subset (Mode 3) for the default and a compact small-data configuration; FS effect in percentage points with bootstrap CI over datasets.",
               "tab:r32_ft_fs")
    print(D_TABLE.round(2).to_string(index=False))

## Export

In [ ]:
# ==========================================================================
# EXPORT: zip all outputs (raw CSVs, tables, figures) next to OUT_ROOT
# ==========================================================================
shutil.make_archive(os.path.join(os.path.dirname(OUT_ROOT), os.path.basename(OUT_ROOT) + "_outputs"), "zip", OUT_ROOT)
print("All outputs zipped next to", OUT_ROOT)